# SQL Portfolio — QA Analytics Database
## 15 Queries demostrando SQL avanzado con datos reales

**Author:** Pedro Medina | Data Analyst  
**Database:** SQLite (qa_evaluations.db)  
**Dataset:** 50,100 evaluaciones QA — 2026  

---

### Técnicas SQL demostradas
| Técnica | Queries |
|---------|----------|
| Agregaciones & GROUP BY | Q01, Q02, Q03, Q08 |
| CTEs (Common Table Expressions) | Q04, Q05, Q07, Q09, Q13, Q14, Q15 |
| Window Functions (RANK, ROW_NUMBER, LAG, LEAD, NTILE) | Q04, Q05, Q06, Q12, Q14, Q15 |
| CASE WHEN (condicionales & bucketing) | Q02, Q03, Q09, Q11, Q13 |
| Subqueries & HAVING | Q07, Q10 |
| Date Functions (strftime) | Q06, Q11, Q14 |
| UNION ALL | Q05 |

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
import subprocess, sys

ROOT   = Path('..').resolve()
DB_PATH = ROOT / 'sql' / 'qa_evaluations.db'

# Crear la DB si no existe
if not DB_PATH.exists():
    print('Creating database...')
    subprocess.run([sys.executable, str(ROOT / 'sql' / 'create_db.py')], check=True)

# Función helper para ejecutar SQL y retornar DataFrame
def run_sql(query: str, db: Path = DB_PATH) -> pd.DataFrame:
    with sqlite3.connect(db) as conn:
        return pd.read_sql_query(query, conn)

print(f'✅ Connected to: {DB_PATH.name}')
run_sql("SELECT COUNT(*) AS total_records, COUNT(DISTINCT Agent) AS agents, "
        "COUNT(DISTINCT Team) AS teams, COUNT(DISTINCT Process) AS processes "
        "FROM evaluations WHERE data_valid=1")

---
## Q01 — KPI Executive Overview
**Propósito:** Vista global de todos los KPIs en una sola query.  
**Técnica:** Múltiples agregaciones, ROUND, CAST, CASE WHEN inline.

In [ ]:
q01 = """
SELECT
    COUNT(*)                                                        AS Total_Evaluations,
    ROUND(AVG(Evaluation_Score), 2)                                 AS Avg_QA_Score,
    ROUND(100.0 * SUM(CASE WHEN Status = 'Pass' THEN 1 ELSE 0 END)
          / COUNT(*), 2)                                            AS Pass_Rate_Pct,
    ROUND(100.0 * SUM(CASE WHEN Critical_Error = 'Yes' THEN 1 ELSE 0 END)
          / COUNT(*), 2)                                            AS Critical_Error_Rate_Pct,
    ROUND(AVG(Customer_Satisfaction), 2)                            AS Avg_CSAT,
    ROUND(AVG(Resolution_Time_Min), 2)                              AS Avg_Resolution_Min,
    MIN(Date)                                                       AS Period_Start,
    MAX(Date)                                                       AS Period_End
FROM evaluations
WHERE data_valid = 1
"""
run_sql(q01)

---
## Q02 — Team Performance Dashboard
**Técnica:** GROUP BY + múltiples métricas + CASE WHEN para nivel de desempeño.

In [ ]:
q02 = """
SELECT
    Team,
    COUNT(*) AS Evaluations,
    ROUND(AVG(Evaluation_Score), 2) AS Avg_QA_Score,
    ROUND(100.0 * SUM(CASE WHEN Status='Pass' THEN 1 ELSE 0 END)/COUNT(*),2) AS Pass_Rate,
    ROUND(100.0 * SUM(CASE WHEN Critical_Error='Yes' THEN 1 ELSE 0 END)/COUNT(*),2) AS Crit_Error_Rate,
    ROUND(AVG(Customer_Satisfaction), 2) AS Avg_CSAT,
    CASE
        WHEN AVG(Evaluation_Score) >= 90 THEN '✅ Excellent'
        WHEN AVG(Evaluation_Score) >= 85 THEN '🟡 On Target'
        ELSE '🔴 Below Target'
    END AS Status
FROM evaluations
WHERE data_valid = 1
GROUP BY Team
ORDER BY Avg_QA_Score DESC
"""
run_sql(q02)

---
## Q04 — Agent Ranking with Window Functions
**Técnica:** CTE + `RANK()` + `AVG() OVER()` para comparar vs promedio global + ranking por equipo.

In [ ]:
q04 = """
WITH AgentStats AS (
    SELECT Agent, Team, Supervisor,
        COUNT(*) AS Evals,
        ROUND(AVG(Evaluation_Score),2) AS Avg_QA,
        ROUND(100.0*SUM(CASE WHEN Status='Pass' THEN 1 ELSE 0 END)/COUNT(*),2) AS Pass_Rate,
        ROUND(AVG(Customer_Satisfaction),2) AS Avg_CSAT
    FROM evaluations
    WHERE data_valid=1
    GROUP BY Agent, Team, Supervisor
    HAVING Evals >= 30
)
SELECT
    Agent, Team, Evals, Avg_QA, Pass_Rate, Avg_CSAT,
    RANK() OVER (ORDER BY Avg_QA DESC)                   AS Global_Rank,
    RANK() OVER (PARTITION BY Team ORDER BY Avg_QA DESC) AS Team_Rank,
    ROUND(AVG(Avg_QA) OVER (), 2)                        AS Global_Avg,
    ROUND(Avg_QA - AVG(Avg_QA) OVER (), 2)               AS Vs_Global_Avg
FROM AgentStats
ORDER BY Global_Rank
LIMIT 20
"""
run_sql(q04)

---
## Q06 — Monthly Trend with MoM Change (LAG)
**Técnica:** `LAG()` para calcular el cambio mes a mes en QA Score y CSAT.  
**Por qué importa:** El promedio global oculta si la calidad está mejorando o deteriorando.

In [ ]:
q06 = """
WITH MonthlyKPIs AS (
    SELECT
        strftime('%Y-%m', Date) AS Period,
        COUNT(*) AS Evals,
        ROUND(AVG(Evaluation_Score),2) AS Avg_QA,
        ROUND(100.0*SUM(CASE WHEN Status='Pass' THEN 1 ELSE 0 END)/COUNT(*),2) AS Pass_Rate,
        ROUND(100.0*SUM(CASE WHEN Critical_Error='Yes' THEN 1 ELSE 0 END)/COUNT(*),2) AS Crit_Rate,
        ROUND(AVG(Customer_Satisfaction),2) AS Avg_CSAT
    FROM evaluations WHERE data_valid=1
    GROUP BY Period
)
SELECT
    Period, Evals, Avg_QA,
    LAG(Avg_QA) OVER (ORDER BY Period)                        AS Prev_QA,
    ROUND(Avg_QA - LAG(Avg_QA) OVER (ORDER BY Period), 2)    AS QA_MoM,
    Pass_Rate, Crit_Rate, Avg_CSAT,
    ROUND(Avg_CSAT - LAG(Avg_CSAT) OVER (ORDER BY Period),2) AS CSAT_MoM
FROM MonthlyKPIs
ORDER BY Period
"""
df06 = run_sql(q06)
df06

---
## Q14 — Running Cumulative Pass Rate
**Técnica:** `SUM() OVER (ORDER BY)` para calcular acumulados progresivos.

In [ ]:
q14 = """
WITH Monthly AS (
    SELECT strftime('%Y-%m', Date) AS Period,
        COUNT(*) AS Evals,
        SUM(CASE WHEN Status='Pass' THEN 1 ELSE 0 END) AS Passed
    FROM evaluations WHERE data_valid=1
    GROUP BY Period
)
SELECT
    Period, Evals, Passed,
    SUM(Evals)   OVER (ORDER BY Period) AS Cumulative_Evals,
    SUM(Passed)  OVER (ORDER BY Period) AS Cumulative_Pass,
    ROUND(100.0 * SUM(Passed) OVER (ORDER BY Period)
               / SUM(Evals)  OVER (ORDER BY Period), 2) AS Cumulative_Pass_Rate
FROM Monthly ORDER BY Period
"""
run_sql(q14)

---
## Q15 — Agent Improvement Detection (LEAD/LAG by Agent)
**Técnica:** `LAG() OVER (PARTITION BY Agent)` para comparar períodos del mismo agente.  
**Por qué importa:** Detectar si un agente está mejorando o empeorando en el tiempo.

In [ ]:
q15 = """
WITH AgentMonthly AS (
    SELECT Agent, Team, strftime('%Y-%m', Date) AS Period,
        COUNT(*) AS Evals,
        ROUND(AVG(Evaluation_Score),2) AS Avg_QA
    FROM evaluations WHERE data_valid=1
    GROUP BY Agent, Team, Period HAVING Evals>=5
),
WithLag AS (
    SELECT *, LAG(Avg_QA) OVER (PARTITION BY Agent ORDER BY Period) AS Prev_QA
    FROM AgentMonthly
)
SELECT Agent, Team, Period, Avg_QA, Prev_QA,
    ROUND(Avg_QA - Prev_QA, 2) AS MoM_Change,
    CASE
        WHEN Prev_QA IS NULL       THEN 'First Period'
        WHEN Avg_QA - Prev_QA > 2 THEN '📈 Improving'
        WHEN Avg_QA - Prev_QA < -2 THEN '📉 Declining'
        ELSE '➡️ Stable'
    END AS Trend
FROM WithLag WHERE Prev_QA IS NOT NULL
ORDER BY ABS(MoM_Change) DESC
LIMIT 20
"""
run_sql(q15)

---
## Resumen de técnicas SQL demostradas

```
✅ CTEs (WITH clause)              — Q04, Q05, Q06, Q07, Q12, Q14, Q15
✅ Window Functions
   - RANK() OVER()                 — Q04
   - RANK() OVER (PARTITION BY)    — Q04  
   - LAG() / LEAD()                — Q06, Q15
   - SUM() OVER (ORDER BY)         — Q14 (running total)
   - NTILE()                       — Q12
   - AVG() OVER ()                 — Q04, Q09
   - ROW_NUMBER()                  — Q05
✅ CASE WHEN (bucketing & labels)  — Q02, Q03, Q09, Q11, Q13
✅ Date Functions strftime()       — Q06, Q11, Q14
✅ UNION ALL                       — Q05
✅ HAVING (filtro post-agregación) — Q04, Q07
✅ Subqueries / Correlated queries — Q10
```

> **Nota:** Estas mismas queries funcionan en PostgreSQL, SQL Server y MySQL con cambios mínimos en las funciones de fecha.